# 05 · Train one row  (step 10-12, GPU session)
Pick a config and seed. Checkpoints every epoch; `--max-hours` stops cleanly before Kaggle's 12 h limit and
the same command resumes. Main rows: baseline, B_uniform, A_uniform, A_agree × seeds 0, 1, 2 (minimum 2).
Ablations: `configs/ablations/*.yaml`, seed 0. Two T4s: run two configs at once with
`CUDA_VISIBLE_DEVICES=0` / `=1` in two background processes.

In [ ]:
# --- setup: code + upstream PEdger -------------------------------------------
import os, subprocess, sys
REPO_URL = ""          # e.g. "https://github.com/<you>/pedger-mtkd.git" (or attach the code as a Kaggle Dataset)
CODE_DATASET = "/kaggle/input/pedger-mtkd-code"
W = "/kaggle/working"
REPO = f"{W}/pedger-mtkd"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["cp", "-r", CODE_DATASET, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py
!pip install -q -r requirements.txt

In [ ]:
# --- inputs (attach these Kaggle Datasets; names are suggestions) --------------
RAW_BSDS = "/kaggle/input/bsds500-raw/BSR/BSDS500/data"     # original Berkeley release
DATA = "/kaggle/input/pedger-data"                           # output of 01 (array stores), saved as a Dataset
CACHE = "/kaggle/input/pedger-teachers"                      # output of 03 (teacher cache), saved as a Dataset
RUNS = "/kaggle/working/runs"
import os
if not os.path.exists(DATA): DATA = "/kaggle/working/data_store"
if not os.path.exists(CACHE): CACHE = "/kaggle/working/teacher_cache"

In [ ]:
CONFIG, SEED = "configs/A_agree.yaml", 0
NAME = os.path.splitext(os.path.basename(CONFIG))[0]
PREV = "/kaggle/input/pedger-runs"          # optional: previous session's runs/ to resume from
if os.path.exists(f"{PREV}/{NAME}_s{SEED}"):
    !mkdir -p {RUNS} && cp -r {PREV}/{NAME}_s{SEED} {RUNS}/
!python -m pedger_kd.train_kd --config {CONFIG} --seed {SEED} --data-root {DATA} --teacher-root {CACHE} --out {RUNS}/{NAME}_s{SEED} --max-hours 11.5

In [ ]:
# Optional: two runs in parallel on Kaggle's 2xT4
# !CUDA_VISIBLE_DEVICES=0 nohup python -m pedger_kd.train_kd --config configs/baseline.yaml --seed 1 --data-root {DATA} --teacher-root {CACHE} --out {RUNS}/baseline_s1 --max-hours 11.5 > {RUNS}/baseline_s1.log 2>&1 &
# !CUDA_VISIBLE_DEVICES=1 nohup python -m pedger_kd.train_kd --config configs/A_agree.yaml --seed 1 --data-root {DATA} --teacher-root {CACHE} --out {RUNS}/A_agree_s1 --max-hours 11.5 > {RUNS}/A_agree_s1.log 2>&1 &